# Phase 4: Training

**Goal:** configure and run the training loop with **`SFTTrainer`**. It handles the loop, logging, checkpointing, and evaluation hooks.

Assumes **`model`**, **`tokenizer`**, and a formatted **`train_dataset`** from earlier phases exist in your session (e.g. you ran Phases 2–3 in order in the same kernel).

## Imports

- **`SFTTrainer`:** supervised fine-tuning from TRL.
- **`TrainingArguments`:** learning rate, batch size, precision, checkpoints, etc.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

## 1. Training hyperparameters (`TrainingArguments`)

| Argument | Role |
|----------|------|
| **`output_dir`** | Where checkpoints are written |
| **`num_train_epochs`** | Full passes over the dataset (often 1–3) |
| **`per_device_train_batch_size`** | Examples per GPU step (start small: 1–2) |
| **`gradient_accumulation_steps`** | Effective batch = this × per-device batch |
| **`learning_rate`** | Optimizer step size (~`2e-4` is a common LoRA start) |
| **`warmup_steps`** | LR ramp-up for stability |
| **`lr_scheduler_type`** | How LR changes (`"cosine"` is standard) |
| **`logging_steps`** | Log loss every N steps |
| **`save_steps`** | Checkpoint every N steps |
| **`bf16` / `fp16`** | Half-precision training. **`bf16=True`** on Ampere+ (A100, RTX 3090+); **`fp16=True`** on older GPUs (T4, V100). |

Fill in the `???` placeholders below.

In [ ]:
training_args = TrainingArguments(
    output_dir="./phi3-finetuned",
    num_train_epochs=???,
    per_device_train_batch_size=???,
    gradient_accumulation_steps=???,
    learning_rate=???,
    warmup_steps=???,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=100,
    bf16=???,
)

## 2. Build `SFTTrainer`

- **`model`:** LoRA-wrapped model from Phase 3.
- **`train_dataset`:** formatted dataset from Phase 2.
- **`dataset_text_field`:** column with chat-formatted strings (e.g. `"text"`).
- **`max_seq_length`:** truncate longer sequences (≤ 4096 for Phi-3 mini 4k; below uses 2048 as a practical default).

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=???,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=2048,
    args=training_args,
)

## 3. Run training

Watch **loss**: it should trend down. Flat or rising loss often means LR too high, bad data, or a chat-template mismatch.

In [ ]:
trainer.train()